In [1]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


In [2]:
## loading model from dotenv

import os
from dotenv import load_dotenv,dotenv_values
load_dotenv()

m2 = os.getenv("MODEL_3")
print(m2)

llama3.1:8b


In [3]:
#model = OllamaLLM(model="llama3.1:8b")    ##directly use model name without dotenv
model = OllamaLLM(model=m2)

In [54]:
template = """
You are an expert in recommendation for the restaurant named "Pizza Place" and thus will be asked about various type of restaurants and where the best type of particular food will be available, etc.
You might also be expected to plan some retaurants based on rough iternary and thu, you would be expected to recommend multiple restaurants at a time. 

Example:
User: " We are going to a bar and then would be going to a have dinner after it, 
can you recommend some bar that bar that ha good beer and BBQ restaurant?"
Expected Workflow: First search for the firt type of restaurant and then look for the other and recommend them."

If the user asks for a particular style of pizza/dish, search for the pizza or dish that fits the description and include its name in the result. 
If you cannot find the appropriate result, inform the user about it.

Example: 
User: Do they have any vegan pizza.
Agent-action: 1.) first search for the required pizza, in this case you got a result with name "tropical blast".
If you also obtain the name of the dish/ pizza, be sure to include the name of the pizza/dish in the final result.


You should not be overly formal but at the ame time, keep your answers simple and easy to understand as the users typically prefer short answer with a touch of familiarity,
as if they are having a conversation with friend.
Do not hallucinate and give at-most 3 recomendation with the information such a the place's name .

Here are some relevant reviews: {reviews}

Here is the question to answer: {questions}
"""
prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

##test
chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":"What is the best pizza place in town?"})

"You're looking for the best pizza place in town, huh? Based on the reviews, I'd say Lafamilia is the top pick! They're known for serving amazing pizzas, and it seems like they're a local favorite. If you're craving a great pizza, I'd recommend checking them out!"

In [4]:
## Write in a python script
'''
while True:
    question = input("Ask your question (q to quit):")
    if question == "q":
        break
    
    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})
    
'''

'\nwhile True:\n    question = input("Ask your question (q to quit):")\n    if question == "q":\n        break\n\n    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})\n\n'

In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
import os
import pandas as pd

In [6]:

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
#dir = "data/csv/RRr.csv"
#df = pd.read_csv(dir)

In [13]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./data/pdf")

    

Found 2 PDF files to process

Processing: Titans_Learning_to_Memorize_at_Test_Time.pdf
  ✓ Loaded 27 pages

Processing: VLA models overview.pdf
  ✓ Loaded 28 pages

Total documents loaded: 55


In [14]:
all_pdf_documents[0:5]

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-03T01:44:50+00:00', 'author': '', 'keywords': '', 'moddate': '2025-01-03T01:44:50+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\Titans_Learning_to_Memorize_at_Test_Time.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'source_file': 'Titans_Learning_to_Memorize_at_Test_Time.pdf', 'file_type': 'pdf'}, page_content='Titans: Learning to Memorize at Test Time\nAli Behrouz\n†\n, Peilin Zhong\n†\n, and Vahab Mirrokni\n†\n†\nGoogle Research\n{alibehrouz, peilinz, mirrokni}@google.com\nAbstract\nOver more than a decade there has been an extensive research effort of how effectively utilize recurrent models and\nattentions. While recurrent models aim to compress the data into a fixed-size memory (called hidden state), attention allows\nattendi

In [15]:
### Text Splitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,length_function=len,separators=["\n\n", "\n", " ", ""])
    
    ## Alternate splitting
    """
    all_chunks = []
    for doc in documents:
        try:
            chunks = text_splitter.split_text(doc.page_content)
            for i, chunk in enumerate(chunks):
                new_doc = doc.copy()
                new_doc.page_content = chunk
                new_doc.metadata['chunk_index'] = i
                all_chunks.append(new_doc)
            print(f"  ✓ Split document into {len(chunks)} chunks")
        except Exception as e:
            print(f"  ✗ Error splitting document: {e}")
    
    print(f"\nTotal chunks created: {len(all_chunks)}")
    return all_chunks
    """
    
    split_docs = text_splitter.split_documents(documents)
    print(f"\nTotal chunks created: {len(split_docs)}")
          
    if split_docs:
        print(f"  ✓ Successfully split documents into chunks" )
        print(f"  ✓ Sample chunk metadata: {split_docs[0].metadata}")
        print(f"  ✓ Sample chunk content: {split_docs[0].page_content[:200]}...")  # Print first 200 chars of the first chunk
    else:
        print(f"  ✗ No chunks were created")
        
    return split_docs

In [16]:
DocChunks = split_documents(all_pdf_documents)
DocChunks


Total chunks created: 343
  ✓ Successfully split documents into chunks
  ✓ Sample chunk metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-03T01:44:50+00:00', 'author': '', 'keywords': '', 'moddate': '2025-01-03T01:44:50+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\Titans_Learning_to_Memorize_at_Test_Time.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'source_file': 'Titans_Learning_to_Memorize_at_Test_Time.pdf', 'file_type': 'pdf'}
  ✓ Sample chunk content: Titans: Learning to Memorize at Test Time
Ali Behrouz
†
, Peilin Zhong
†
, and Vahab Mirrokni
†
†
Google Research
{alibehrouz, peilinz, mirrokni}@google.com
Abstract
Over more than a decade there has ...


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-01-03T01:44:50+00:00', 'author': '', 'keywords': '', 'moddate': '2025-01-03T01:44:50+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\\pdf\\Titans_Learning_to_Memorize_at_Test_Time.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'source_file': 'Titans_Learning_to_Memorize_at_Test_Time.pdf', 'file_type': 'pdf'}, page_content='Titans: Learning to Memorize at Test Time\nAli Behrouz\n†\n, Peilin Zhong\n†\n, and Vahab Mirrokni\n†\n†\nGoogle Research\n{alibehrouz, peilinz, mirrokni}@google.com\nAbstract\nOver more than a decade there has been an extensive research effort of how effectively utilize recurrent models and\nattentions. While recurrent models aim to compress the data into a fixed-size memory (called hidden state), attention allows\nattendi

In [25]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:v1")

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "mxbai-embed-large:v1"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = OllamaEmbeddings(model=self.model_name)
            print(f"Model loaded successfully. ")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager  

Loading embedding model: mxbai-embed-large:v1
Model loaded successfully. 


In [28]:
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
"""
db_location = "./chrome_langchain_db"
add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    ids = []
    
    for i, row in df.iterrows():
        document = Document(
            page_content = row["Title"] + " " + row["Review"],
            metadata = {"rating": row["Rating"], "date":row["Date"]},
            id= str(i)
        )
        ids.append(str(i))
        documents.append(document)
    
    
vector_store = Chroma(
    collection_name = "restaurant_reviews",
    persist_directory=db_location,
    embedding_function = embeddings
)
        
if add_documents:
    vector_store.add_documents(documents=documents,ids = ids)
"""

'\ndb_location = "./chrome_langchain_db"\nadd_documents = not os.path.exists(db_location)\n\nif add_documents:\n    documents = []\n    ids = []\n\n    for i, row in df.iterrows():\n        document = Document(\n            page_content = row["Title"] + " " + row["Review"],\n            metadata = {"rating": row["Rating"], "date":row["Date"]},\n            id= str(i)\n        )\n        ids.append(str(i))\n        documents.append(document)\n\n\nvector_store = Chroma(\n    collection_name = "restaurant_reviews",\n    persist_directory=db_location,\n    embedding_function = embeddings\n)\n\nif add_documents:\n    vector_store.add_documents(documents=documents,ids = ids)\n'

In [29]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [30]:
### Convert the text to embeddings
texts=[doc.page_content for doc in DocChunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(DocChunks,embeddings)

Generating embeddings for 343 texts...


AttributeError: 'OllamaEmbeddings' object has no attribute 'encode'

In [60]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k":5}
)

In [61]:
import re
regex_pattern = r"<think>[\s\S]*?<\/think>\n?\n?"


In [62]:
question = "Do they have a roman style pizza?"

reviews = retriever.invoke(question)

result = chain.invoke({"reviews":reviews,"questions":question})
#cleaned_response = re.sub(regex_pattern, '', result)
print(result) # Output: This is the actual answer


Based on the reviews, I found a mention of Roman-style pizza in review #1. The reviewer mentioned that the Roman-style pizza al taglio (rectangular, with a focaccia-like base) is a welcome alternative to the usual round pies.

So, yes! They do have a Roman-style pizza. 

Recommendation:
1. Try their Roman-style pizza al taglio.
2. You can also consider their other options, like the wood-fired Margherita pizza (review #2).
3. If you want something a bit different, go for their fig and prosciutto pizza, which has been mentioned as excellent in review #4.
